# DESCRIPTION
Notebook for running the PUREdrop hardware 

# 1 - SETUP
The following steps set up all the controllers, check if all connections are valid, and set all instruments to their starting positions and settings. 

In [1]:
# importing the necesary python modules
from acqpack.acqpack import New_Mfcs
from acqpack.acqpack import New_Motor, New_AsiController, Autosampler, FractionCollector
from acqpack.acqpack import log
import pandas as pd
import time

### 1a - Define your experiment
Here, all parameters for this run are set. That is, it is provided where the .yaml and .txt files are that are needed, plus it is set where the logfile should be stored. 

In [2]:
# Pump
path_to_pump_configuration = 'acqpack/Experiments/exp/config/mfcs/mfcs.yaml'
path_to_channel_map = 'acqpack/Experiments/exp/config/mfcs/chanmap.txt'

# Autosampler
path_to_motor_configuration = 'acqpack/Experiments/exp/config/sampler/motor.yaml' 
path_to_asi_configuration = 'acqpack/Experiments/exp/config/sampler/asi.yaml'
path_to_sampler_deck = 'acqpack/run/config/sampler/deck.txt' 
path_to_sampler_plate = 'acqpack/Experiments/exp/config/sampler/96platePoC.txt' 

# Fraction Collector
path_to_collector_configuration = 'acqpack/Experiments/exp/config/collector/asi.yaml' 
path_to_collector_deck = 'acqpack/run/config/collector/deck.txt' 
path_to_collector_plate = 'acqpack/Experiments/exp/config/collector/96platePoC-coldplate.txt' 

# Logging
path_to_logdir = 'acqpack/Experiments/exp/log' 

### 1b - Initiate the Logger
The logger helps you logging what you did and making sure all info on the experiment is saved. 
Above, in the *path_to_logdir* variable, you defined where the directory containing al logged information will be. If this already exists, it will throw an Error. You can overwrite existing logging directories via

*logger = log.MyLogger(path_to_logdir, overwrite=True)*

If you want to manually log something from within this Notebook, you can do so via

*logger.log('your text')*

If you want to save a file to the logging directory from this Notebook, you can do so via

*logger.savefile('path_to_file_to_save')*

In [ ]:
logger = log.MyLogger(path_to_logdir, overwrite=True)

### 1c - Initiating all controllers and positioning sampler & collector
The controllers to all the instruments are set up. The autosampler and fraction collector will then move to their starting positions. All this is automated, you do not have to do anything. 

In [ ]:
# initiate pump, valves and sensors (Fluigent)
#logger = log.MyLogger('./logging/test/')
p = New_Mfcs(path_to_pump_configuration, path_to_channel_map, logger)
p.chanmap

In [ ]:
# initiate sampler
a = Autosampler(New_Motor(path_to_motor_configuration, logger=logger),  New_AsiController(path_to_asi_configuration, logger, logger_name='Sampler'))

In [ ]:
# initiate fraction collector
f = FractionCollector(New_AsiController(path_to_collector_configuration, logger, logger_name='FractionCollector'))
f.set_velocity(30,30) #adjusting stage speed
print(f.get_velocity()) 

In [7]:
# Setting up config files
a.add_frame('plate', path_to_sampler_deck, path_to_sampler_plate)
a.zh_travel = 45 # hardware safe travel height ## TODO should this be moved to 1a? do you mean the whole initiate or just the travel height? 
f.add_frame('plate', path_to_collector_deck, path_to_collector_plate)

In [ ]:
# Move sampler to buffer/blank well
a.goto('plate','well','A07',45)
logger.log('sampler at Blank', True)

In [ ]:
f.goto('plate','xy',(105,83))
logger.log('collector at Waste', True)

In [ ]:
# list switches connected
p.get_switch_options()

In [ ]:
# list sensors connected
p.get_sensor_options()

### 1d - Manual setup and adjustment of pressures
Now, you have to manually set the desired pressures based on external visual feedback. Use the Oxygen software

1. Mount valves into chip individually and wait until the dead channels are filled with water
2. Mount main outlet and IAsampler outlet
3. Mount IAsampler and IA2 into chip

In [ ]:
#   Test priming
a.goto('plate','well','C04',45)
p.set_switch_position(0,4) #waste reservoir
p.set_switch_position(1,0) #waste channel open
p.set_switch_position(5,1) #2IA channels closed
for k, v in p_table.loc['prime'].items(): #apply priming pressures
        p.set('name', k, v)
time.sleep(30)   
p.set('name','IAsampler', 0)
p.set_switch_position(1,1)

In [ ]:
p.set('name','IAsampler',1500) 
time.sleep(30)
p.set('name','IAsampler',0) 

connect the OIL into chip 
- 255 um diameter tubing, 
- 35 um height chip

Adjust pressures to get ~10 ul/min stable production

Time to measure
1. Total flow rate 
2. Droplet size 
3. Generation frequency

In [ ]:
closed_pressure = 1.0
p_table = pd.DataFrame(columns=['IAsampler','OIL','pure', 'Valve2' ], index=['prime','drops','flush', 'movement']).fillna(closed_pressure) 

p_table.loc['prime','IAsampler'] = 1500 
p_table.loc['prime','OIL'] = 0  
p_table.loc['prime','pure'] = 0 
p_table.loc['prime','Valve2'] = 4250

p_table.loc['drops','IAsampler'] = 1250 
p_table.loc['drops','OIL'] = 3100 
p_table.loc['drops','pure'] = 1250 
p_table.loc['drops','Valve2'] = 4250 

p_table.loc['movement','IAsampler'] = 0 
p_table.loc['movement','OIL'] = 0  
p_table.loc['movement','pure'] = 0 
p_table.loc['movement','Valve2'] = 4250 

p_table.loc['flush','IAsampler'] = 0 
p_table.loc['flush','OIL'] = 0  
p_table.loc['flush','pure'] = 0 
p_table.loc['flush','Valve2'] = 0 

p_table

Droplet generator testing: Pausing, Playing and cleraning

In [13]:
#Play drops
p.set_switch_position(1,1) #waste valve closed
p.set_switch_position(5,0) #2IA valve closed   
p.set('name','IAsampler', 300)
p.set('name','pure', 300)
time.sleep(0.7) 
for k, v in p_table.loc['drops'].items():
        p.set('name', k, v)
time.sleep(2)     
sensor_index1 = 0
sensor_index2 = 1
setpoint1 = 0.28 
setpoint2 = 0.52 
p.set_sensor_regulation_lookup(sensor_index1, 'name','IAsampler', setpoint1)
p.set_sensor_regulation_lookup(sensor_index2, 'name','pure', setpoint2)

In [14]:
# Pause drops
p.set('name','OIL', 0)
time.sleep(0.3) 
for k, v in p_table.loc['movement'].items(): #moving 
        p.set('name', k, v)
p.set_switch_position(1,1) #waste valve closed
p.set_switch_position(5,1) #2IA valve closed

In [ ]:
#   testing clean up
p.set_switch_position(5,1) #2IA channel closed
for k, v in p_table.loc['movement'].items(): #moving 
    p.set('name', k, v)
time.sleep(1)
p.set_switch_position(5,1) #2IA channel closed  
a.goto('plate','xyz',(0,35,20),5) # waste outside plate
p.set_switch_position(1,0) #waste channel open
p.set('name','Water', 1500)
p.set_switch_position(0,5) #ethanol
time.sleep(60) #time for water wash at 3000 mbar 
p.set_switch_position(0,6) #water
time.sleep(60) #time for water wash at 3000 mbar 
p.set('name','Water', 0)
p.set('name','OIL', 0)
p.set_switch_position(0,4) #waste reservoir
p.set_switch_position(5,1) #2IA channel closed  
p.set_switch_position(1,1) #waste channel closed
a.goto('plate','xyz',(22.5,35,5),5)

### 1e - Reading in the maps
Here, the program reads in from the tsv (?) files that contain the information about what is where on your 96-well plates. 

In [ ]:
platemap = a.frames['plate'].position_table
codemap = platemap.loc[platemap['series'].notnull()].reset_index(drop=True)  # rows of platemap that have a code
codemap

# SCRIPT

In [14]:
# PRIME: sampler_n -> chip (includes oil wash between wells)  
def prime(code, sec_P2C, total_Oil): #P2S: plate to switch
    logger.log('Priming liquid before switch', True) #log to initiate priming
    p.set('name','OIL', 0)
    time.sleep(0.5) #time for valves to actuate
    for k, v in p_table.loc['movement'].items(): #moving 
        p.set('name', k, v)
    p.set_switch_position(5,1) #2IA channel closed
    a.goto('plate','xyz',(0,35,20),5) # waste outside plate
    p.set_switch_position(1,0) #waste channel open
    p.set_switch_position(0,5) #ethanol
    p.set('name','Water', 1500)
    p.set('name','OIL', 3000)
    Oil_start_time = time.time()
    time.sleep(70) #time for water wash at 1500 mbar 
    p.set_switch_position(0,6) #water
    time.sleep(70) #time for water wash at 3000 mbar 
    p.set('name','Water', 0)
    p.set_switch_position(0,4) #waste reservoir
    p.set_switch_position(1,1) #waste channel closed
    a.goto('plate','xyz',(20,35,5),5)
    a.goto('plate', 'n', code['n'], zh_travel=45) #sampler travel from blank well to sample well
    logger.log('sampler at plate::well={} c={}'.format(code['well'], code['code']), True) #log movement
    p.set('name','Valve1', 0)
    p.set_switch_position(1,0) #waste channel open
    time.sleep(1) 
    for k, v in p_table.loc['prime'].items(): #apply priming pressures
        p.set('name', k, v)  
    logger.log('{} PRIME {} {}'.format(code.name, code['well'], code['code']), True) #log priming 
    p.set('name','OIL', 3000)
    time.sleep(sec_P2C) #hold pressure 100s to to push 25ul through 
    p.set('name','IAsampler', 0)
    p.set_switch_position(1,1) #waste channel closed
    Oil_int_time = time.time()
    time_elapsed_Oil = Oil_int_time - Oil_start_time
    time.sleep(total_Oil - time_elapsed_Oil)     
    p.set('name','OIL', 0)
    time.sleep(1)

In [15]:
# DROPS: sampler_n -> collector 
def drops(code, vial, sec_production):  #S2C: switch to chip - C2W: chip to wells 
    p.set_switch_position(5,0)  
    p.set('name','IAsampler', 300)
    p.set('name','pure', 300)
    time.sleep(0.7)  
    for k, v in p_table.loc['drops'].items():
        p.set('name', k, v)
    time.sleep(2) 
    sensor_index1 = 0
    sensor_index2 = 1
    setpoint1 = 0.28 
    setpoint2 = 0.52 
    p.set_sensor_regulation_lookup(sensor_index1, 'name','IAsampler', setpoint1)
    p.set_sensor_regulation_lookup(sensor_index2, 'name','pure', setpoint2)
    logger.log('{} DROPS {} {}'.format(code.name, code['well'], code['code']), True) # log
    time.sleep(sec_production/2) 
    # go to next well
    f.goto('plate', 'vial', vial)
    logger.log('collector at plate::vial={} c={}'.format(vial, code['code']), True) #log collecting drops
    time.sleep(sec_production/2)

In [16]:
# SHUTDOWN: After run (empty liquid path)
def shutdown():
    logger.log('shutting down', True)
    p.set('name','OIL', 0)
    time.sleep(1.5) #time for valves to actuate
    for k, v in p_table.loc['movement'].items(): #moving 
        p.set('name', k, v)
    p.set_switch_position(1,1) #waste valve open
    p.set_switch_position(5,1) #2IA valve closed

    f.goto('plate','xy',(68,59)) #go to waste to empty well
    logger.log('collector at Waste', True)
    a.goto('plate','xyz',(59.5,33,0),5) #sampler home
    logger.log('sampler at home', True)  
    
    p_table.to_csv('pressures.csv', sep='\t')  # pressures
    f.frames['plate'].position_table.to_csv('collected_plate.csv', index=False, sep='\t')  # collection record
    pd.DataFrame(l).to_csv('log.txt', index=False, header=False) # log
    logger.log('DONE', True)

# 3: Run your experiment!

In [17]:
# set the times
sec_P2C = 35 
sec_C2W = 240
sec_production = 100  
total_Oil = sec_C2W 

In [ ]:
# RUN!!!
l = []  
vial_start = 1 # starting collector vial

for i, code in codemap.iterrows():
    vial = vial_start + i  # code.name = i; vial where the fc will collect
    prime(code, sec_P2C, total_Oil)
    drops(code, vial, sec_production) 
  
p.set('name','OIL', 0)
time.sleep(0.5) #time for valves to actuate
for k, v in p_table.loc['movement'].items(): #moving 
    p.set('name', k, v)
p.set_switch_position(5,1) #2IA channel closed
time.sleep(1) #time for valves to actuate    
p.set('name','OIL', 3000)
time.sleep(total_Oil)     
p.set('name','OIL', 0)
shutdown()